# Train external validation model


In [ ]:
import gc
import os
import numpy as np
import pandas as pd
import anndata as ad
from tqdm import tqdm
from functools import partial
from datetime import datetime
from multiprocessing import Pool, cpu_count
from joblib import dump
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve, auc, precision_recall_curve
import warnings
warnings.filterwarnings('ignore')

OUTPUT_DIR = 'results/classification_sle_external/results_elasticnet_C1_l1_0.5'

In [ ]:
adata = ad.read_h5ad("data/adata_cohort1.h5ad")
print("UCSF total samples:", adata.shape)
print(adata.obs['group'].value_counts())

nyu = ad.read_h5ad("data/adata_cohort3.h5ad")
nyu_control_ids = set(nyu.obs.loc[nyu.obs['group'] == 'control', 'subject_id'].astype(str))
ucsf_hc_ids = set(adata.obs.loc[adata.obs['group'] == 'healthy_control', 'subject_id'].astype(str))
shared_hc_ids = ucsf_hc_ids & nyu_control_ids

print(f"\nUCSF HC donors overlapping with NYU controls: {len(shared_hc_ids)} (expected 46)")

adata = adata[~adata.obs['subject_id'].astype(str).isin(shared_hc_ids)].copy()
print("\nAfter excluding shared HC donors:", adata.shape)
print(adata.obs['group'].value_counts())

In [ ]:
def donor_based_cv_split(X_train, y_train, adata, n_splits=5, random_state=42):
    train_donors = np.array(adata[adata.obs.index.isin(X_train.index)].obs.unique_patient_id.unique())

    np.random.seed(random_state)
    np.random.shuffle(train_donors)
    fold_size = len(train_donors) // n_splits
    folds = []

    for i in range(n_splits):
        if i == n_splits - 1:
            val_donors = train_donors[i*fold_size:]
        else:
            val_donors = train_donors[i*fold_size:(i+1)*fold_size]

        train_donors_fold = np.array([d for d in train_donors if d not in val_donors])

        train_indices = X_train.index[adata[adata.obs.index.isin(X_train.index)]
                                    .obs.unique_patient_id.isin(train_donors_fold)].tolist()
        val_indices = X_train.index[adata[adata.obs.index.isin(X_train.index)]
                                  .obs.unique_patient_id.isin(val_donors)].tolist()

        folds.append((train_indices, val_indices))

    return folds

In [ ]:
def analyze_single_iteration_with_roc(seed, adata, model_type='elastic_net',
                                     C=1.0, l1_ratio=0.5, n_splits=5):
    gc.collect()

    adjusted_adata_foldchange_df_filter = adata.to_df(layer="adjusted_fc_over_ag")
    X_filter = adjusted_adata_foldchange_df_filter.copy()
    HC_samples = np.array(adata[adata.obs.group==('healthy_control')].obs.index)
    y_filter = ~(X_filter.index.isin(HC_samples))
    y_filter = np.array(y_filter)
    X_filter[np.isnan(X_filter) | np.isinf(X_filter)] = 0

    n_pos = sum(y_filter)
    n_neg = len(y_filter) - n_pos
    class_ratio = n_pos / n_neg
    class_weights = {0: 1, 1: 1/class_ratio} if class_ratio < 1 else {0: class_ratio, 1: 1}

    cv_folds = donor_based_cv_split(X_filter, y_filter, adata,
                                   n_splits=n_splits, random_state=seed)

    iteration_results, models, roc_data = [], [], []

    for fold_idx, (train_idx, val_idx) in enumerate(cv_folds):
        X_fold_train = X_filter.loc[train_idx]
        X_fold_val = X_filter.loc[val_idx]
        train_mask = X_filter.index.isin(train_idx)
        val_mask = X_filter.index.isin(val_idx)
        y_fold_train = y_filter[train_mask]
        y_fold_val = y_filter[val_mask]

        if model_type == 'ridge':
            model = LogisticRegression(
                penalty='l2', solver='lbfgs', C=C,
                class_weight=class_weights, max_iter=500,
                random_state=seed + fold_idx, n_jobs=1)
        elif model_type == 'lasso':
            model = LogisticRegression(
                penalty='l1', solver='liblinear', C=C,
                class_weight=class_weights, max_iter=500,
                random_state=seed + fold_idx)
        elif model_type == 'elastic_net':
            model = LogisticRegression(
                penalty='elasticnet', solver='saga',
                l1_ratio=l1_ratio, C=C,
                class_weight=class_weights,
                max_iter=100, tol=1e-3,
                random_state=seed + fold_idx,
                n_jobs=1)

        model.fit(X_fold_train, y_fold_train)
        y_pred_proba = model.predict_proba(X_fold_val)[:, 1]

        fpr, tpr, thresholds = roc_curve(y_fold_val, y_pred_proba)
        prec, rec, pr_thresholds = precision_recall_curve(y_fold_val, y_pred_proba)

        j_scores = tpr - fpr
        optimal_idx = np.argmax(j_scores)
        optimal_threshold = thresholds[optimal_idx]

        iteration_results.append({
            'seed': seed, 'fold': fold_idx, 'optimal_threshold': optimal_threshold,
            'auroc': roc_auc_score(y_fold_val, y_pred_proba),
            'auprc': auc(rec, prec),
            'n_features': np.sum(model.coef_[0] != 0),
            'n_train_samples': len(X_fold_train),
            'n_test_samples': len(X_fold_val),
            'n_train_donors': len(set(adata[adata.obs.index.isin(train_idx)].obs.unique_patient_id)),
            'n_test_donors': len(set(adata[adata.obs.index.isin(val_idx)].obs.unique_patient_id))
        })

        models.append(model)

        roc_data.append({
            'seed': seed, 'fold': fold_idx, 'fpr': fpr, 'tpr': tpr,
            'thresholds': thresholds, 'y_true': y_fold_val,
            'y_pred_proba': y_pred_proba})

    feature_names = list(X_filter.columns)

    del X_filter
    gc.collect()

    return iteration_results, models, roc_data, feature_names

def run_batched_cv_analysis(adata, output_dir, model_type='elastic_net',
                           C=1.0, l1_ratio=0.5, n_iterations=20,
                           batch_size=5, n_jobs=5):
    os.makedirs(output_dir, exist_ok=True)

    print(f"\n{'='*60}")
    print(f"Model Configuration:")
    print(f"Type: {model_type}")
    print(f"C: {C}")
    if model_type == 'elastic_net':
        print(f"l1_ratio: {l1_ratio}")
    print(f"Iterations: {n_iterations}, Batch size: {batch_size}, Jobs: {n_jobs}")
    print(f"Output directory: {output_dir}")
    print(f"{'='*60}\n")

    all_results = []
    all_models = {}
    all_roc_data = []
    feature_names = None

    n_batches = (n_iterations + batch_size - 1) // batch_size

    for batch in tqdm(range(n_batches), desc="Processing batches"):
        start_seed = batch * batch_size
        end_seed = min((batch + 1) * batch_size, n_iterations)
        seeds = [42 + i * 1000 for i in range(start_seed, end_seed)]

        print(f"\nBatch {batch+1}/{n_batches}: Processing iterations {start_seed}-{end_seed-1}")

        analyze_func = partial(
            analyze_single_iteration_with_roc,
            adata=adata, model_type=model_type,
            C=C, l1_ratio=l1_ratio, n_splits=5)

        with Pool(n_jobs) as pool:
            batch_results = list(pool.imap(analyze_func, seeds))

        for iteration_idx, (seed, results) in enumerate(zip(seeds, batch_results)):
            iter_results, models, roc_data, features = results

            if feature_names is None:
                feature_names = features

            all_results.extend(iter_results)

            for fold_idx, model in enumerate(models):
                all_models[f"seed_{seed}_fold_{fold_idx}"] = model

            all_roc_data.extend(roc_data)

        temp_df = pd.DataFrame(all_results)
        temp_df.to_csv(f'{output_dir}/results_temp.csv', index=False)

        dump(all_models, f'{output_dir}/models_batch_{batch+1}.joblib')
        dump(all_roc_data, f'{output_dir}/roc_data_batch_{batch+1}.joblib')
        gc.collect()

    results_df = pd.DataFrame(all_results)

    print("\nSaving final results...")
    results_df.to_csv(os.path.join(output_dir, 'results.csv'), index=False)
    dump(all_models, os.path.join(output_dir, 'models.joblib'))
    dump(all_roc_data, os.path.join(output_dir, 'roc_data.joblib'))

    with open(os.path.join(output_dir, 'feature_names.txt'), 'w') as f:
        f.write('\n'.join(feature_names))

    config = {
        'model_type': model_type, 'C': C,
        'l1_ratio': l1_ratio if model_type == 'elastic_net' else None,
        'n_iterations': n_iterations, 'n_splits': 5,
        'batch_size': batch_size, 'n_jobs': n_jobs,
        'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
    dump(config, os.path.join(output_dir, 'config.joblib'))

    if os.path.exists(f'{output_dir}/results_temp.csv'):
        os.remove(f'{output_dir}/results_temp.csv')
    for batch in range(n_batches):
        for file_type in ['models', 'roc_data']:
            batch_file = f'{output_dir}/{file_type}_batch_{batch+1}.joblib'
            if os.path.exists(batch_file):
                os.remove(batch_file)

    print(f"\nResults Summary:")
    print(f"AUROC: {results_df['auroc'].mean():.4f} ± {results_df['auroc'].std():.4f}")
    print(f"AUPRC: {results_df['auprc'].mean():.4f} ± {results_df['auprc'].std():.4f}")
    print(f"Features: {results_df['n_features'].mean():.0f} ± {results_df['n_features'].std():.0f}")
    print(f"Min features: {results_df['n_features'].min()}")
    print(f"Max features: {results_df['n_features'].max()}")

    return {
        'results_df': results_df, 'models': all_models,
        'roc_data': all_roc_data, 'feature_names': feature_names,
        'config': config}

In [ ]:
n_jobs = 10
print(f"Using n_jobs={n_jobs} ({cpu_count()} CPUs available)")

results = run_batched_cv_analysis(
    adata,
    output_dir=OUTPUT_DIR,
    model_type='elastic_net', C=1, l1_ratio=0.5,
    n_iterations=20, batch_size=20,
    n_jobs=n_jobs)

In [ ]:
def analyze_feature_selection(models, features, threshold=1e-8):
    coef_matrix = [model.coef_[0] for model in models.values()]
    coef_array = np.array(coef_matrix)

    nonzero_mask = np.abs(coef_array) > threshold
    positive_mask = coef_array > threshold
    negative_mask = coef_array < -threshold

    selection_freq = np.mean(nonzero_mask, axis=0)
    positive_freq = np.mean(positive_mask, axis=0)
    negative_freq = np.mean(negative_mask, axis=0)

    median_coef = np.zeros(len(features))
    mean_coef = np.zeros(len(features))
    std_coef = np.zeros(len(features))

    for i in range(len(features)):
        selected_coefs = coef_array[nonzero_mask[:, i], i]
        if len(selected_coefs) > 0:
            median_coef[i] = np.median(selected_coefs)
            mean_coef[i] = np.mean(selected_coefs)
            std_coef[i] = np.std(selected_coefs)

    freq_df = pd.DataFrame({
        'peptide': features,
        'selection_frequency': selection_freq,
        'positive_frequency': positive_freq,
        'negative_frequency': negative_freq,
        'median_coefficient': median_coef,
        'mean_coefficient': mean_coef,
        'std_coefficient': std_coef,
        'cv_coefficient': np.divide(std_coef, np.abs(mean_coef),
                                   out=np.zeros_like(std_coef),
                                   where=mean_coef!=0)
    })

    freq_df['sign_consistency'] = np.maximum(
        freq_df['positive_frequency'],
        freq_df['negative_frequency']
    ) / freq_df['selection_frequency'].clip(lower=1e-10)

    freq_df = freq_df.sort_values('selection_frequency', ascending=False)
    return freq_df

def bin_features_by_stability(freq_df, output_dir=None):
    df = freq_df.copy()

    freq_bins = [
        (0.0, 0.5, 'LowFreq'),
        (0.5, 0.9, 'ModerateFreq'),
        (0.9, 1.1, 'HighFreq')
    ]
    mag_bins = [
        (0.0, 1e-3, 'LowMag'),
        (1e-3, 1e-2, 'MedMag'),
        (1e-2, float('inf'), 'HighMag')
    ]

    def assign_bin(value, bins):
        for low, high, label in bins:
            if low <= value < high:
                return label
        return 'Unknown'

    df['freq_bin'] = df['selection_frequency'].apply(lambda x: assign_bin(x, freq_bins))
    df['mag_bin'] = df['median_coefficient'].abs().apply(lambda x: assign_bin(x, mag_bins))
    df['sign'] = df['median_coefficient'].apply(
        lambda x: 'positive' if x > 0 else ('negative' if x < 0 else 'zero')
    )

    crosstab = pd.crosstab([df['freq_bin'], df['mag_bin']], df['sign'])
    print("\nFeature Distribution by Frequency, Magnitude, and Sign:")
    print(crosstab)

    if output_dir:
        df.to_csv(f'{output_dir}/binned_peptides_improved.csv', index=False)
        high_confidence = df[
            (df['freq_bin'] == 'HighFreq') &
            (df['mag_bin'] == 'HighMag')
        ].sort_values('median_coefficient', ascending=False)
        high_confidence.to_csv(f'{output_dir}/high_confidence_peptides.csv', index=False)
        print(f"\nSaved {len(high_confidence)} high-confidence features")
        print(f"Positive: {sum(high_confidence['sign'] == 'positive')}")
        print(f"Negative: {sum(high_confidence['sign'] == 'negative')}")

    return df

freq_df = analyze_feature_selection(results['models'], results['feature_names'])
binned_df = bin_features_by_stability(freq_df, output_dir=OUTPUT_DIR)